# Exploratory Data Analysis (EDA)
## Amazon ML Challenge 2026 — Business Entity Resolution

Structured exploratory analysis of challenge data across Source 1, 2, 3, and Ground Truth.

In [1]:
import pandas as pd
import numpy as np
from rapidfuzz.fuzz import ratio
from tqdm import tqdm

path = "../ML_dataset/student_resource/dataset/train"

s1 = pd.read_csv(f"{path}/train_source1.tsv", sep="\t")
s2 = pd.read_csv(f"{path}/train_source2.tsv", sep="\t")
s3 = pd.read_csv(f"{path}/train_source3.tsv", sep="\t")
source1 = s1
gt = pd.read_csv(f"{path}/train_ground_truth.tsv", sep="\t")
source2 = pd.read_csv(f"{path}/train_source2.tsv", sep="\t")
source3 = pd.read_csv(f"{path}/train_source3.tsv", sep="\t")

In [3]:
datasets = {
    "Source 1": s1,
    "Source 2": s2,
    "Source 3": s3,
    "Ground Truth": gt
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print(df.head(3))


Source 1
Shape: (2206821, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']
      entity_id        business_name                        business_address  \
0  S1-925783039  Orelee's Barbershop  1795 Westchester Drive, High Point, NC   
1  S1-773889195          Prime Money         17560 Ellis Road, Tahlequah, OK   
2  S1-377745466        B+ Retail Inc     1712 Montebello Avenue, Phoenix, AZ   

  country  
0      US  
1      US  
2      US  

Source 2
Shape: (5034616, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']
      entity_id                    business_name  \
0  S2-166376419  राम मार्केटिंग प्राइवेट लिमिटेड   
1  S2-764573417     -- Holloway Peak Inc Seafood   
2  S2-639257739         आदित्य प्रॉपर्टीज एलएलपी   

                                   business_address country  
0      KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi   India  
1                         105 ELM ST, MORGANTON, NC      US  
2  G-3/571, GULMOHAR COLONY, BHOPA

In [9]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isna().sum())
    print(df.isna().mean().mul(100).round(2))


Source 1
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64
entity_id           0.0
business_name       0.0
business_address    0.0
country             0.0
dtype: float64

Source 2
entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64
entity_id           0.00
business_name       0.00
business_address    3.36
country             0.00
dtype: float64

Source 3
entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64
entity_id           0.00
business_name       0.00
business_address    3.33
country             0.00
dtype: float64

Ground Truth
source1_entity_id          0
matched_entity_ids    123247
dtype: int64
source1_entity_id     0.00
matched_entity_ids    5.58
dtype: float64


In [11]:
for name, df in {
    "Source 1": s1,
    "Source 2": s2,
    "Source 3": s3
}.items():
    print(f"\n{name}")
    print("Duplicate rows:", df.duplicated().sum())
    print("Duplicate entity IDs:", df["entity_id"].duplicated().sum())

print("\nGround Truth")
print("Duplicate rows:", gt.duplicated().sum())
print("Duplicate Source-1 IDs:", gt["source1_entity_id"].duplicated().sum())


Source 1
Duplicate rows: 0
Duplicate entity IDs: 0

Source 2
Duplicate rows: 0
Duplicate entity IDs: 0

Source 3
Duplicate rows: 0
Duplicate entity IDs: 0

Ground Truth
Duplicate rows: 0
Duplicate Source-1 IDs: 0


In [12]:
for name, df in {"S1": s1, "S2": s2, "S3": s3}.items():
    print(f"\n{name}")
    print("Duplicate rows:", df.duplicated().sum())
    print("Duplicate IDs:", df["entity_id"].duplicated().sum())
    
    print("Duplicate names:", df["business_name"].duplicated().sum())


S1
Duplicate rows: 0
Duplicate IDs: 0
Duplicate names: 667592

S2
Duplicate rows: 0
Duplicate IDs: 0
Duplicate names: 632607

S3
Duplicate rows: 0
Duplicate IDs: 0
Duplicate names: 633994


In [13]:
for name, df in {"S1": s1, "S2": s2, "S3": s3}.items():
    counts = df["business_name"].value_counts()
    print(f"{name}:")
    print("Unique names:", counts.size)
    print("Names appearing >1 time:", (counts > 1).sum())
    print("Most common names:")
    print(counts.head(5))
    print()

S1:
Unique names: 1539229
Names appearing >1 time: 177793
Most common names:
business_name
Primary Care Group         253
Ear Nose & Throat Group    251
Pediatric Group            222
Womens Health Group        220
Physical Therapy Group     218
Name: count, dtype: int64

S2:
Unique names: 4402008
Names appearing >1 time: 239778
Most common names:
business_name
Primary Care         320
Physical Therapy     307
Urgent Care          297
Womens Health        297
Behavioral Health    296
Name: count, dtype: int64

S3:
Unique names: 4651608
Names appearing >1 time: 258275
Most common names:
business_name
Primary Care        421
Physical Therapy    399
Pediatric Dental    393
Womens Health       379
Urgent Care         377
Name: count, dtype: int64



In [14]:
for name, df in {"S1": s1, "S2": s2, "S3": s3}.items():
    print(f"\n{name}")
    print(df["country"].value_counts().head(20))


S1
country
US       1323633
India     883188
Name: count, dtype: int64

S2
country
US       3016817
India    2017799
Name: count, dtype: int64

S3
country
US       3170056
India    2115547
Name: count, dtype: int64


In [15]:
for name, df in {"S1": s1, "S2": s2, "S3": s3}.items():
    length = df["business_address"].fillna("").str.len()

    print(f"\n{name}")
    print(length.describe())


S1
count    2.206821e+06
mean     5.206621e+01
std      2.532972e+01
min      1.100000e+01
25%      3.300000e+01
50%      4.100000e+01
75%      7.000000e+01
max      2.560000e+02
Name: business_address, dtype: float64

S2
count    5.034616e+06
mean     4.622591e+01
std      2.484467e+01
min      0.000000e+00
25%      3.000000e+01
50%      3.700000e+01
75%      6.100000e+01
max      2.490000e+02
Name: business_address, dtype: float64

S3
count    5.285603e+06
mean     4.671444e+01
std      2.164981e+01
min      0.000000e+00
25%      3.500000e+01
50%      4.200000e+01
75%      5.400000e+01
max      2.400000e+02
Name: business_address, dtype: float64


In [16]:
for name, df in {"S1": s1, "S2": s2, "S3": s3}.items():
    length = df["business_name"].fillna("").str.len()

    print(f"\n{name}")
    print(length.describe())


S1
count    2.206821e+06
mean     2.403440e+01
std      7.740533e+00
min      3.000000e+00
25%      1.800000e+01
50%      2.400000e+01
75%      3.000000e+01
max      1.050000e+02
Name: business_name, dtype: float64

S2
count    5.034616e+06
mean     2.510357e+01
std      8.894694e+00
min      0.000000e+00
25%      1.900000e+01
50%      2.500000e+01
75%      3.100000e+01
max      1.040000e+02
Name: business_name, dtype: float64

S3
count    5.285603e+06
mean     2.520214e+01
std      9.493635e+00
min      0.000000e+00
25%      1.800000e+01
50%      2.500000e+01
75%      3.100000e+01
max      1.230000e+02
Name: business_name, dtype: float64


In [17]:
matches = gt["matched_entity_ids"].fillna("")

match_count = matches.apply(
    lambda x: 0 if x == "" else len(x.split(","))
)

match_count.describe()

count    2.206821e+06
mean     3.461253e+00
std      1.705323e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: matched_entity_ids, dtype: float64

In [18]:
match_count.value_counts().sort_index()

matched_entity_ids
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64

In [21]:
print("Jupyter is working")

Jupyter is working


In [22]:
gt["matched_entity_ids"].head(10)

0    S2-681193310,S2-743505751,S3-775321672,S3-1129...
1    S2-249013014,S2-197070651,S3-478195123,S3-3843...
2               S2-790675320,S2-479876582,S3-878454467
3                S2-153058913,S2-24659151,S3-679606215
4    S2-478959098,S2-553508714,S2-625774905,S3-7280...
5    S2-755677256,S3-187831601,S3-641489370,S3-4762...
6                            S2-660036492,S3-804600254
7                            S3-274817120,S3-312496301
8                            S2-648035184,S3-588502663
9                S2-383871912,S3-74481402,S3-576451439
Name: matched_entity_ids, dtype: object

In [23]:
print(gt["source1_entity_id"].isin(s1["entity_id"]).mean())

1.0


In [24]:
print(gt["source1_entity_id"].isin(s1["entity_id"]).value_counts())

source1_entity_id
True    2206821
Name: count, dtype: int64


In [25]:
sample_ids = gt["matched_entity_ids"].dropna().head(100)

sample_ids

0      S2-681193310,S2-743505751,S3-775321672,S3-1129...
1      S2-249013014,S2-197070651,S3-478195123,S3-3843...
2                 S2-790675320,S2-479876582,S3-878454467
3                  S2-153058913,S2-24659151,S3-679606215
4      S2-478959098,S2-553508714,S2-625774905,S3-7280...
                             ...                        
101    S2-718026833,S2-752280842,S3-255051180,S3-6903...
102    S2-955993024,S2-339477476,S2-981272489,S3-9637...
103    S2-26500789,S2-912528411,S3-741223656,S3-64382...
104                                         S2-198723291
105               S2-138434584,S2-180445273,S2-598796961
Name: matched_entity_ids, Length: 100, dtype: object

In [27]:
s2 = gt["matched_entity_ids"].str.contains("S2-", na=False)
s3 = gt["matched_entity_ids"].str.contains("S3-", na=False)

print("S2 only:", (s2 & ~s3).sum())
print("S3 only:", (s3 & ~s2).sum())
print("Both:", (s2 & s3).sum())
print("No match:", (~s2 & ~s3).sum())

S2 only: 143029
S3 only: 164498
Both: 1776047
No match: 123247


In [28]:
s2_count = gt["matched_entity_ids"].str.count("S2-").fillna(0)
s3_count = gt["matched_entity_ids"].str.count("S3-").fillna(0)

print("S2 matches per S1:")
print(s2_count.describe())

print("\nS3 matches per S1:")
print(s3_count.describe())

S2 matches per S1:
count    2.206821e+06
mean     1.673728e+00
std      1.112813e+00
min      0.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      2.000000e+00
max      5.000000e+00
Name: matched_entity_ids, dtype: float64

S3 matches per S1:
count    2.206821e+06
mean     1.787524e+00
std      1.165604e+00
min      0.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      6.000000e+00
Name: matched_entity_ids, dtype: float64


In [29]:
print("\nS2 count distribution:")
print(s2_count.value_counts().sort_index())

print("\nS3 count distribution:")
print(s3_count.value_counts().sort_index())


S2 count distribution:
matched_entity_ids
0.0    287745
1.0    789108
2.0    652779
3.0    333957
4.0    119078
5.0     24154
Name: count, dtype: int64

S3 count distribution:
matched_entity_ids
0.0    266276
1.0    716417
2.0    668375
3.0    372443
4.0    145116
5.0     35378
6.0      2816
Name: count, dtype: int64


# Phase 1 — Exploratory Data Analysis (EDA)

## 1. Dataset Overview

The training dataset contains three source datasets and one ground-truth file.

| Dataset | Rows | Columns |
|---|---:|---:|
| Source 1 | 2,206,821 | 4 |
| Source 2 | 5,034,616 | 4 |
| Source 3 | 5,285,603 | 4 |
| Ground Truth | 2,206,821 | 2 |

### Source columns

- `entity_id`
- `business_name`
- `business_address`
- `country`

### Ground Truth columns

- `source1_entity_id`
- `matched_entity_ids`

`matched_entity_ids` contains comma-separated IDs from Source 2 and/or Source 3 that represent the same real-world entity as the Source-1 entity.

---

## 2. Missing Value Analysis

| Dataset | Missing Business Name | Missing Address |
|---|---:|---:|
| Source 1 | 0 | 0 |
| Source 2 | 2 (≈0%) | 168,967 (3.36%) |
| Source 3 | 13 (≈0%) | 175,916 (3.33%) |

Ground Truth contains **123,247 empty match entries (5.58%)**.

These represent Source-1 entities with **no matching entity** in Source 2 or Source 3, rather than ordinary missing data.

### Finding

Source 1 is complete, while approximately 3.3% of Source 2 and Source 3 addresses are unavailable. Therefore, address information cannot always be relied upon during entity matching.

---

## 3. Duplicate Analysis

### Entity-level duplicates

- Source 1: 0 duplicate rows, 0 duplicate entity IDs
- Source 2: 0 duplicate rows, 0 duplicate entity IDs
- Source 3: 0 duplicate rows, 0 duplicate entity IDs
- Ground Truth: 0 duplicate rows

However, business names are frequently repeated.

| Source | Duplicate Business Names |
|---|---:|
| Source 1 | 667,592 |
| Source 2 | 632,607 |
| Source 3 | 633,994 |

### Finding

`business_name` is **not a unique identifier**.

Examples of highly repeated names include:

- `Primary Care Group`
- `Ear Nose & Throat Group`
- `Physical Therapy`
- `Urgent Care`

Therefore, matching using business name alone can produce many false candidates.

---

## 4. Business Name Distribution

| Source | Unique Names | Names Appearing More Than Once |
|---|---:|---:|
| Source 1 | 1,539,229 | 177,793 |
| Source 2 | 4,402,008 | 239,778 |
| Source 3 | 4,651,608 | 258,275 |

Average business-name length:

| Source | Mean Length |
|---|---:|
| Source 1 | 24.0 |
| Source 2 | 25.1 |
| Source 3 | 25.2 |

### Finding

Business-name lengths are similar across all three sources, but repeated/generic names are common. Name similarity will therefore need to be combined with other attributes.

---

## 5. Country Distribution

| Source | US | India |
|---|---:|---:|
| Source 1 | 1,323,633 | 883,188 |
| Source 2 | 3,016,817 | 2,017,799 |
| Source 3 | 3,170,056 | 2,115,547 |

All three sources have approximately the same distribution:

- ~60% US
- ~40% India

### Finding

`country` is consistent across the three sources and can potentially be used as a strong blocking/filtering attribute during candidate generation.

---

## 6. Address Analysis

Average address length:

| Source | Mean | Median | Minimum | Maximum |
|---|---:|---:|---:|---:|
| Source 1 | 52.1 | 41 | 11 | 256 |
| Source 2 | 46.2 | 37 | 0 | 249 |
| Source 3 | 46.7 | 42 | 0 | 240 |

### Finding

Source 1 has no missing addresses, while Source 2 and Source 3 contain missing addresses.

Address information is therefore useful for matching, but cannot be treated as universally available.

---

## 7. Ground Truth — Number of Matches per Source-1 Entity

Each Source-1 entity has one Ground Truth row.

Number of matches ranges from **0 to 11**.

| Number of Matches | Source-1 Entities |
|---:|---:|
| 0 | 123,247 |
| 1 | 119,157 |
| 2 | 375,212 |
| 3 | 530,841 |
| 4 | 484,115 |
| 5 | 321,957 |
| 6 | 164,868 |
| 7 | 63,968 |
| 8 | 18,680 |
| 9 | 4,205 |
| 10 | 534 |
| 11 | 37 |

### Finding

This is a **one-to-many entity matching problem**, not a simple one-to-one matching problem.

- 123,247 (~5.58%) have no match.
- The most common case is 3 matches.
- Most entities have 2–5 matches.
- Maximum observed matches = 11.

---

## 8. Source-2 vs Source-3 Match Distribution

| Match Type | Source-1 Entities |
|---|---:|
| S2 only | 143,029 |
| S3 only | 164,498 |
| Both S2 and S3 | 1,776,047 |
| No match | 123,247 |

### Finding

Approximately **80.5%** of Source-1 entities have matches in **both Source 2 and Source 3**.

This indicates that information from both sources can potentially provide complementary evidence for entity matching.

---

## 9. Matches per Source

### Source 2

| Matches per S1 | Entities |
|---:|---:|
| 0 | 287,745 |
| 1 | 789,108 |
| 2 | 652,779 |
| 3 | 333,957 |
| 4 | 119,078 |
| 5 | 24,154 |

Mean = **1.67 matches/S1**  
Median = **2**  
Maximum = **5**

### Source 3

| Matches per S1 | Entities |
|---:|---:|
| 0 | 266,276 |
| 1 | 716,417 |
| 2 | 668,375 |
| 3 | 372,443 |
| 4 | 145,116 |
| 5 | 35,378 |
| 6 | 2,816 |

Mean = **1.79 matches/S1**  
Median = **2**  
Maximum = **6**

### Finding

Source 3 contributes slightly more matches per Source-1 entity on average than Source 2.

Typical contribution:

- Source 2 → 1–2 matches
- Source 3 → 1–3 matches

---

## 10. Ground Truth Integrity Check

All Ground Truth `source1_entity_id` values were verified against Source 1.

```text
2,206,821 / 2,206,821 = 100%

In [30]:
sample_gt = gt.head(10000)

sample_gt.head()

,source1_entity_id,matched_entity_ids,s2_matches,s3_matches
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129...",2,3
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843...",2,2
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467",2,1
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215",2,1
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280...",3,3


In [32]:
row = gt.iloc[0]

print("S1 ID:", row["source1_entity_id"])
print("Matches:", row["matched_entity_ids"])

S1 ID: S1-965667
Matches: S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364


In [33]:
s1[s1["entity_id"] == "S1-965667"]

,entity_id,business_name,business_address,country
1286323,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US


In [37]:
ids = [
    "S2-681193310", "S2-743505751",
    "S3-775321672", "S3-11291185", "S3-860443364"
]

pd.concat([
    source2[source2["entity_id"].isin(ids)],
    source3[source3["entity_id"].isin(ids)]
])

,entity_id,business_name,business_address,country
1376739,S2-681193310,Maure Wilblims Colombier Inc,NaN,US
2003754,S2-743505751,Maure Williams Colombier,NaN,US
63830,S3-860443364,Maure Williams Inc Center,NaN,US
1217885,S3-775321672,Dréxkor,"85 Wanye Avenue, Ticonderoga Townshiip, New York",US
5223525,S3-11291185,maurewilliamscolombier.com,"Wayne Ave, Ticonderoga Townshiip, New York",US


#Major EDA finding

Exact string matching will not be sufficient.

In [39]:
source1 = s1
source2 = source2
source3 = source3

In [40]:
source1[source1["entity_id"] == "S1-965667"]

,entity_id,business_name,business_address,country
1286323,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US


# now  below we are doing random choices 

In [42]:
import random

sample = gt.sample(10, random_state=42)

for _, row in sample.iterrows():
    print("\n" + "="*80)
    print("S1:", row["source1_entity_id"])
    print("Matches:", row["matched_entity_ids"])

    display(
        source1[source1["entity_id"] == row["source1_entity_id"]]
    )

    match_string = row["matched_entity_ids"]
    ids = match_string.split(",") if pd.notna(match_string) and match_string != "" else []

    display(pd.concat([
        source2[source2["entity_id"].isin(ids)],
        source3[source3["entity_id"].isin(ids)]
    ]))


S1: S1-384734976
Matches: S2-685146894,S2-908470079,S3-109845102,S3-176459320


,entity_id,business_name,business_address,country
1645812,S1-384734976,"FQ Social, Inc","1212 Old Rte 34, Sandwich, IL",US


,entity_id,business_name,business_address,country
564407,S2-685146894,"FQ Social, Inc.","1212 OLD RTE 34, SANDWICH, IL",US
585013,S2-908470079,"FQ Social, #42836","1212 OLD RTE 34, SANDWICH, IL",US
1795329,S3-109845102,"FQ Social, Ínc","1212 Old Rte 34, Sandwich, Illinois",US
4635538,S3-176459320,"Arcxylo DBA FQ Social, Inc","1212 Old Rte 34, Sandwich, IL",US



S1: S1-574038342
Matches: S2-133116927,S2-180265553,S2-201976425


,entity_id,business_name,business_address,country
612837,S1-574038342,Innovative Consultants,"15 16 Indrapuri Madhubancolony Karvenagar, Pun...",India


...

,entity_id,business_name,business_address,country
520132,S1-281641466,Life Construction Private Limited,"House No- 24/153, 3Rd Floor Blk-24, Delhi, Nor...",India


,entity_id,business_name,business_address,country
1388626,S2-559704787,लाइफ कंस्ट्रक्शन प्राइवेट लिमिटेड,"HOUSE NO- 24/153, DELHI, NORTH DELHI, Delhi",India
1757245,S3-772912089,Shri Life Construction Private,"House No- 24/153, 3Rd Floor Blk-24, Delhi, Nor...",India



S1: S1-608038725
Matches: S2-895472957,S2-192681436,S2-217803628,S3-993567829


,entity_id,business_name,business_address,country
473783,S1-608038725,Anchor Staking Inc,"2330 Central Avenue, Kansas City, KS",US


,entity_id,business_name,business_address,country
358703,S2-192681436,anchor staking inc,"2330 CETRAL AVENUE, KANSAS CITY, KS",US
1660665,S2-895472957,Anchor Stáking Inc,"CENTRAL AVE, KANSAS CITY, KS",US
1951598,S2-217803628,ANCHOR STAKIFNG INC,"2330 CENTRAL AVE, KANSAS CITY, KS",US
761061,S3-993567829,Anchorstaking.Com,"#2330 Central Ave, Kansas City, Kansas",US


# Pattern found:

### Name variations — typos, case, punctuation, suffixes.

### Address variations — abbreviations, ordering, spelling errors.

### Missing addresses — especially S2/S3.
### Domain names can represent businesses.
### Multilingual names occur.
### Country is consistent and useful as a constraint.
### One S1 → multiple S2/S3 matches is common.
### Some true matches have weak name similarity but strong address similarity.

In [11]:
from rapidfuzz.fuzz import ratio
from tqdm import tqdm

s1_lookup = source1.set_index("entity_id")["business_name"].to_dict()
s2_lookup = source2.set_index("entity_id")["business_name"].to_dict()
s3_lookup = source3.set_index("entity_id")["business_name"].to_dict()

sample = gt[
    gt["matched_entity_ids"].notna() &
    (gt["matched_entity_ids"].str.strip() != "")
].sample(
    10000,
    random_state=42
)

results = []

for _, row in tqdm(
    sample.iterrows(),
    total=len(sample),
    desc="Calculating name similarities"
):

    s1_id = row["source1_entity_id"]

    s1_name = s1_lookup.get(s1_id)

    if s1_name is None:
        continue

    for match_id in str(row["matched_entity_ids"]).split(","):

        match_id = match_id.strip()

        if match_id.startswith("S2-"):
            name = s2_lookup.get(match_id)
        else:
            name = s3_lookup.get(match_id)

        if name is not None:
            results.append(
                ratio(str(s1_name).lower(), str(name).lower())
            )



print("\nFinished!")
print("Number of matched pairs:", len(results))
print("Mean name similarity:", sum(results) / len(results))
print("Minimum:", min(results))
print("Maximum:", max(results))

Calculating name similarities: 100%|███████████████████████████████████████████| 10000/10000 [00:02<00:00, 4108.28it/s]


Finished!
Number of matched pairs: 36703
Mean name similarity: 77.96565971561394
Minimum: 3.8461538461538436
Maximum: 100.0


In [52]:


print("Name Similarity Distribution")
print("=" * 40)

for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"P{p:02d}: {np.percentile(results, p):.2f}")

print(f"\nMean   : {np.mean(results):.2f}")
print(f"Median : {np.median(results):.2f}")
print(f"Min    : {np.min(results):.2f}")
print(f"Max    : {np.max(results):.2f}")

Name Similarity Distribution
P01: 8.22
P05: 10.53
P10: 44.44
P25: 71.43
P50: 86.21
P75: 94.74
P90: 100.00
P95: 100.00
P99: 100.00

Mean   : 77.97
Median : 86.21
Min    : 3.85
Max    : 100.00


In [53]:
thresholds = [50, 60, 70, 75, 80, 85, 90, 95, 100]

print("Threshold → % of true matches retained")
print("=" * 45)

for t in thresholds:
    retained = sum(x >= t for x in results)
    percentage = retained / len(results) * 100

    print(f">= {t:3d}: {percentage:6.2f}%")

Threshold → % of true matches retained
>=  50:  89.29%
>=  60:  85.34%
>=  70:  76.74%
>=  75:  70.49%
>=  80:  63.20%
>=  85:  52.84%
>=  90:  40.52%
>=  95:  24.28%
>= 100:  10.48%


In [54]:
thresholds = [50, 60, 70, 80, 85, 90, 95, 99, 100]

print("True-match retention by threshold")
print("=" * 45)

for t in thresholds:
    retained = sum(x >= t for x in results)
    percentage = retained / len(results) * 100

    print(
        f">= {t:3d}: "
        f"{retained:6d} / {len(results)} "
        f"({percentage:6.2f}%)"
    )

True-match retention by threshold
>=  50:  32771 / 36703 ( 89.29%)
>=  60:  31323 / 36703 ( 85.34%)
>=  70:  28165 / 36703 ( 76.74%)
>=  80:  23198 / 36703 ( 63.20%)
>=  85:  19393 / 36703 ( 52.84%)
>=  90:  14872 / 36703 ( 40.52%)
>=  95:   8912 / 36703 ( 24.28%)
>=  99:   3846 / 36703 ( 10.48%)
>= 100:   3845 / 36703 ( 10.48%)


# Negative pair analysis

In [57]:
from rapidfuzz.fuzz import ratio
from tqdm import tqdm
import random
import pandas as pd

random.seed(42)

# Take only 20,000 random target rows
s2_sample = source2.sample(20000, random_state=42)
s3_sample = source3.sample(20000, random_state=43)

# Combine targets
targets = pd.concat([
    s2_sample[["entity_id", "business_name", "country"]],
    s3_sample[["entity_id", "business_name", "country"]]
], ignore_index=True)

# Ground-truth lookup only for sampled S1 entities
gt_lookup = gt.set_index("source1_entity_id")["matched_entity_ids"].to_dict()

negative_results = []

# Sample 10k S1 entities
s1_sample = source1.sample(10000, random_state=42)

for _, row in tqdm(
    s1_sample.iterrows(),
    total=len(s1_sample),
    desc="Calculating negative similarities"
):

    # Same-country candidates
    candidates = targets[targets["country"] == row["country"]]

    if len(candidates) == 0:
        continue

    candidate = candidates.sample(1).iloc[0]

    # Make sure it's not a known match
    true_ids = str(gt_lookup.get(row["entity_id"], ""))

    if candidate["entity_id"] in true_ids.split(","):
        continue

    negative_results.append(
        ratio(
            str(row["business_name"]).lower(),
            str(candidate["business_name"]).lower()
        )
    )

print("\nFinished!")
print("Negative pairs:", len(negative_results))
print(
    "Mean negative similarity:",
    sum(negative_results) / len(negative_results)
)

Calculating negative similarities: 100%|██████████████████████████████████████████████████████| 10000/10000 [02:00<00:00, 83.11it/s]


Finished!
Negative pairs: 10000
Mean negative similarity: 32.43143823204454


In [8]:
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

N_NEGATIVES = 10_000
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

print("Starting memory-efficient negative pair generation...")


# ============================================================
# 1. GET COUNTRY INDICES
#    IMPORTANT: We DO NOT create filtered DataFrames.
# ============================================================

s2_us_idx = source2.index[
    source2["country"].eq("US")
].to_numpy()

s2_india_idx = source2.index[
    source2["country"].eq("India")
].to_numpy()

s3_us_idx = source3.index[
    source3["country"].eq("US")
].to_numpy()

s3_india_idx = source3.index[
    source3["country"].eq("India")
].to_numpy()


print("\nCountry index sizes:")
print("S2 US:", len(s2_us_idx))
print("S2 India:", len(s2_india_idx))
print("S3 US:", len(s3_us_idx))
print("S3 India:", len(s3_india_idx))


# ============================================================
# 2. DETERMINE HOW MANY NEGATIVES TO GENERATE
# ============================================================

n = min(
    N_NEGATIVES,
    len(source2),
    len(source3)
)

print(f"\nGenerating {n:,} negative pairs...")


# ============================================================
# 3. SAMPLE ROW INDICES
# ============================================================

# We generate approximately half US and half India
n_us = n // 2
n_india = n - n_us


# ------------------------------------------------------------
# US
# ------------------------------------------------------------

us_s2 = rng.choice(
    s2_us_idx,
    size=min(n_us, len(s2_us_idx)),
    replace=False
)

us_s3 = rng.choice(
    s3_us_idx,
    size=min(n_us, len(s3_us_idx)),
    replace=False
)


# ------------------------------------------------------------
# India
# ------------------------------------------------------------

india_s2 = rng.choice(
    s2_india_idx,
    size=min(n_india, len(s2_india_idx)),
    replace=False
)

india_s3 = rng.choice(
    s3_india_idx,
    size=min(n_india, len(s3_india_idx)),
    replace=False
)


# ============================================================
# 4. COMBINE INDICES
# ============================================================

s2_indices = np.concatenate([
    us_s2,
    india_s2
])

s3_indices = np.concatenate([
    us_s3,
    india_s3
])


# ============================================================
# 5. SHUFFLE PAIRINGS
#    This ensures S2 and S3 rows are not simply aligned.
# ============================================================

rng.shuffle(s3_indices)


# ============================================================
# 6. BUILD NEGATIVE PAIRS
#    ONLY THE SMALL RESULT IS COPIED.
# ============================================================

negative_pairs = pd.DataFrame({
    "s2_idx": s2_indices,
    "s3_idx": s3_indices
})


# ============================================================
# 7. VERIFY COUNTRY MATCHING
# ============================================================

negative_pairs["s2_country"] = source2.loc[
    negative_pairs["s2_idx"],
    "country"
].to_numpy()

negative_pairs["s3_country"] = source3.loc[
    negative_pairs["s3_idx"],
    "country"
].to_numpy()


negative_pairs["same_country"] = (
    negative_pairs["s2_country"]
    == negative_pairs["s3_country"]
)


# ============================================================
# 8. RESULTS
# ============================================================

print("\n" + "=" * 50)
print("NEGATIVE PAIR GENERATION COMPLETE")
print("=" * 50)

print("Total negative pairs:", len(negative_pairs))

print(
    "Same-country pairs:",
    negative_pairs["same_country"].sum()
)

print(
    "Different-country pairs:",
    (~negative_pairs["same_country"]).sum()
)

print("\nCountry distribution:")
print(
    negative_pairs["s2_country"].value_counts()
)

print("\nSample:")
print(
    negative_pairs.head()
)

Starting memory-efficient negative pair generation...

Country index sizes:
S2 US: 3016817
S2 India: 2017799
S3 US: 3170056
S3 India: 2115547

Generating 10,000 negative pairs...

NEGATIVE PAIR GENERATION COMPLETE
Total negative pairs: 10000
Same-country pairs: 5098
Different-country pairs: 4902

Country distribution:
s2_country
US       5000
India    5000
Name: count, dtype: int64

Sample:
    s2_idx   s3_idx s2_country s3_country  same_country
0  1255452  4713875         US         US          True
1  4123207  4153963         US      India         False
2  4968485  2350049         US         US          True
3   895154  1416733         US      India         False
4   659426  3604030         US         US          True


In [6]:
import numpy as np
from rapidfuzz.fuzz import ratio
from tqdm import tqdm

N_NEGATIVES = 10_000
rng = np.random.default_rng(42)

# Country indices — only integer indices
s2_us = source2.index[source2["country"].eq("US")].to_numpy()
s2_india = source2.index[source2["country"].eq("India")].to_numpy()

s3_us = source3.index[source3["country"].eq("US")].to_numpy()
s3_india = source3.index[source3["country"].eq("India")].to_numpy()

# Sample S1
s1_indices = rng.choice(
    len(source1),
    size=N_NEGATIVES,
    replace=False
)

negative_results = []

for idx in tqdm(
    s1_indices,
    desc="Calculating negative similarities"
):

    country = source1.iloc[idx]["country"]

    # Pick S2 or S3, but SAME COUNTRY
    if rng.random() < 0.5:

        candidates = (
            s2_us if country == "US"
            else s2_india
        )

        target_idx = rng.choice(candidates)
        target_name = source2.iloc[target_idx]["business_name"]

    else:

        candidates = (
            s3_us if country == "US"
            else s3_india
        )

        target_idx = rng.choice(candidates)
        target_name = source3.iloc[target_idx]["business_name"]

    s1_name = source1.iloc[idx]["business_name"]

    negative_results.append(
        ratio(
            str(s1_name).lower(),
            str(target_name).lower()
        )
    )

print("\nFinished!")
print("Negative pairs:", len(negative_results))
print(
    "Mean negative similarity:",
    np.mean(negative_results)
)

Calculating negative similarities: 100%|███████████████████████████████████████| 10000/10000 [00:05<00:00, 1729.30it/s]


Finished!
Negative pairs: 10000
Mean negative similarity: 32.46133516016396


In [12]:
import numpy as np

percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]

print(f"{'Percentile':<12}{'True Match':<15}{'Negative':<15}")
print("-" * 42)

for p in percentiles:
    true_score = np.percentile(results, p)
    neg_score = np.percentile(negative_results, p)

    print(f"P{p:<11}{true_score:<15.2f}{neg_score:<15.2f}")

Percentile  True Match     Negative       
------------------------------------------
P1          8.22           5.00           
P5          10.53          9.52           
P10         44.44          17.64          
P25         71.43          25.93          
P50         86.21          32.00          
P75         94.74          38.46          
P90         100.00         47.46          
P95         100.00         54.55          
P99         100.00         66.67          


# Phase 1 — Exploratory Data Analysis (EDA)

## Dataset Structure

| Dataset | Rows | Columns |
|---|---:|---:|
| Source 1 | 2,206,821 | 4 |
| Source 2 | 5,034,616 | 4 |
| Source 3 | 5,285,603 | 4 |
| Ground Truth | 2,206,821 | 2 |

## Key Findings

### Missing Values
- S1 has no missing business names or addresses.
- S2: 2 missing names, 168,967 missing addresses (3.36%).
- S3: 13 missing names, 175,916 missing addresses (3.33%).
- Ground Truth has 123,247 entities (5.58%) with no known match.

### Duplicates
- No duplicate rows or entity IDs in S1, S2, or S3.
- Business names are heavily duplicated.
- Therefore, `business_name` is not a unique identifier.

### Country
- All sources are approximately 60% US and 40% India.
- `country` is a useful potential blocking signal.

### Address
- S1 addresses are always present.
- S2/S3 have ~3.3% missing addresses.
- Address is useful but cannot always be relied upon.

### Ground Truth
- One-to-many matching problem.
- Each S1 entity has 0–11 valid matches.
- 123,247 S1 entities have no known match.
- 100% of Ground Truth S1 IDs exist in S1.

### Manual True-Match Analysis
Observed variations include:
- spelling errors
- capitalization/punctuation differences
- abbreviations
- company suffix changes
- word additions/removals
- address formatting differences
- partial/missing addresses
- domain names
- multilingual names
- substantially different names

Therefore, exact string matching is insufficient.

---

# Name Similarity Analysis

## True Matches

Sample:
- 10,000 S1 entities
- 36,703 true matched pairs

Using `rapidfuzz.fuzz.ratio`:

| Metric | Score |
|---|---:|
| Mean | 77.97 |
| Median | 86.21 |
| Minimum | 3.85 |
| Maximum | 100.00 |
| P10 | 44.44 |
| P25 | 71.43 |
| P75 | 94.74 |
| P90 | 100.00 |
| P95 | 100.00 |
| P99 | 100.00 |

### True-match retention

| Threshold | Retained |
|---:|---:|
| >= 50 | 89.29% |
| >= 60 | 85.34% |
| >= 70 | 76.74% |
| >= 80 | 63.20% |
| >= 85 | 52.84% |
| >= 90 | 40.52% |
| >= 95 | 24.28% |
| >= 99 | 10.48% |
| = 100 | 10.48% |

## Negative Pairs

10,000 random **same-country** negative pairs were generated from the full S2/S3 populations.

Mean negative similarity:

**32.46**

| Percentile | True Match | Negative |
|---|---:|---:|
| P1 | 8.22 | 5.00 |
| P5 | 10.53 | 9.52 |
| P10 | 44.44 | 17.64 |
| P25 | 71.43 | 25.93 |
| P50 | 86.21 | 32.00 |
| P75 | 94.74 | 38.46 |
| P90 | 100.00 | 47.46 |
| P95 | 100.00 | 54.55 |
| P99 | 100.00 | 66.67 |

## Current Conclusion

Business-name similarity is a strong matching signal, but it is not sufficient by itself.

The eventual matcher should combine:

**Business Name + Address + Country + additional signals**

In [13]:
from rapidfuzz.fuzz import ratio
from tqdm import tqdm
import numpy as np

# Fast lookups for only required columns
s1_address = source1.set_index("entity_id")["business_address"].to_dict()
s2_address = source2.set_index("entity_id")["business_address"].to_dict()
s3_address = source3.set_index("entity_id")["business_address"].to_dict()

address_results = []

for _, row in tqdm(
    sample.iterrows(),
    total=len(sample),
    desc="Calculating address similarities"
):
    s1_id = row["source1_entity_id"]

    s1_addr = s1_address.get(s1_id)

    if pd.isna(s1_addr):
        continue

    for match_id in str(row["matched_entity_ids"]).split(","):

        match_id = match_id.strip()

        if match_id.startswith("S2-"):
            addr = s2_address.get(match_id)
        else:
            addr = s3_address.get(match_id)

        # Skip missing addresses
        if pd.isna(addr):
            continue

        address_results.append(
            ratio(
                str(s1_addr).lower(),
                str(addr).lower()
            )
        )

print("\nFinished!")
print("Matched address pairs:", len(address_results))
print("Mean:", np.mean(address_results))
print("Median:", np.median(address_results))
print("Min:", np.min(address_results))
print("Max:", np.max(address_results))

Calculating address similarities: 100%|████████████████████████████████████████| 10000/10000 [00:02<00:00, 4546.14it/s]


Finished!
Matched address pairs: 35088
Mean: 78.77712802354696
Median: 83.56164383561644
Min: 13.913043478260867
Max: 100.0


In [14]:
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]

print(f"{'Percentile':<12}{'Address Similarity':<20}")
print("-" * 32)

for p in percentiles:
    score = np.percentile(address_results, p)
    print(f"P{p:<11}{score:.2f}")

Percentile  Address Similarity  
--------------------------------
P1          32.96
P5          45.00
P10         51.16
P25         68.49
P50         83.56
P75         92.17
P90         97.83
P95         100.00
P99         100.00


In [16]:
from rapidfuzz.fuzz import ratio
from tqdm import tqdm
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# 10k S1 sample
s1_sample = source1.sample(10_000, random_state=42)

# Country indices
s2_us = source2.index[source2["country"].eq("US")].to_numpy()
s2_india = source2.index[source2["country"].eq("India")].to_numpy()
s3_us = source3.index[source3["country"].eq("US")].to_numpy()
s3_india = source3.index[source3["country"].eq("India")].to_numpy()

negative_address_results = []

for idx in tqdm(
    s1_sample.index,
    total=len(s1_sample),
    desc="Calculating negative address similarities"
):
    s1_row = source1.loc[idx]
    country = s1_row["country"]

    if country == "US":
        candidates = s2_us if rng.random() < 0.5 else s3_us
    else:
        candidates = s2_india if rng.random() < 0.5 else s3_india

    target_idx = rng.choice(candidates)

    target_df = source2 if candidates is s2_us or candidates is s2_india else source3
    target = target_df.loc[target_idx]

    if pd.isna(s1_row["business_address"]) or pd.isna(target["business_address"]):
        continue

    negative_address_results.append(
        ratio(
            str(s1_row["business_address"]).lower(),
            str(target["business_address"]).lower()
        )
    )

print("\nFinished!")
print("Negative address pairs:", len(negative_address_results))
print("Mean:", np.mean(negative_address_results))

Calculating negative address similarities: 100%|███████████████████████████████| 10000/10000 [00:05<00:00, 1997.32it/s]


Finished!
Negative address pairs: 9658
Mean: 35.687699646680755


In [17]:
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]

print(f"{'Percentile':<12}{'True Address':<15}{'Negative Address':<18}")
print("-" * 45)

for p in percentiles:
    true_score = np.percentile(address_results, p)
    neg_score = np.percentile(negative_address_results, p)

    print(f"P{p:<11}{true_score:<15.2f}{neg_score:<18.2f}")

Percentile  True Address   Negative Address  
---------------------------------------------
P1          32.96          21.43             
P5          45.00          26.23             
P10         51.16          28.57             
P25         68.49          32.00             
P50         83.56          35.48             
P75         92.17          39.08             
P90         97.83          43.04             
P95         100.00         45.90             
P99         100.00         52.63             


In [19]:
from rapidfuzz.fuzz import ratio
from tqdm import tqdm
import numpy as np
import pandas as pd

# Lookups
s1_names = source1.set_index("entity_id")["business_name"].to_dict()
s1_addresses = source1.set_index("entity_id")["business_address"].to_dict()

s2_names = source2.set_index("entity_id")["business_name"].to_dict()
s2_addresses = source2.set_index("entity_id")["business_address"].to_dict()

s3_names = source3.set_index("entity_id")["business_name"].to_dict()
s3_addresses = source3.set_index("entity_id")["business_address"].to_dict()

combined_results = []

for _, row in tqdm(
    sample.iterrows(),
    total=len(sample),
    desc="Calculating combined similarity"
):

    s1_id = row["source1_entity_id"]

    s1_name = s1_names.get(s1_id)
    s1_addr = s1_addresses.get(s1_id)

    if pd.isna(s1_name) or pd.isna(s1_addr):
        continue

    for match_id in str(row["matched_entity_ids"]).split(","):

        match_id = match_id.strip()

        if match_id.startswith("S2-"):
            target_name = s2_names.get(match_id)
            target_addr = s2_addresses.get(match_id)
        else:
            target_name = s3_names.get(match_id)
            target_addr = s3_addresses.get(match_id)

        # Address may be missing
        if pd.isna(target_name) or pd.isna(target_addr):
            continue

        name_score = ratio(
            str(s1_name).lower(),
            str(target_name).lower()
        )

        address_score = ratio(
            str(s1_addr).lower(),
            str(target_addr).lower()
        )

        combined_score = (
            0.5 * name_score +
            0.5 * address_score
        )

        combined_results.append(combined_score)

print("\nFinished!")
print("Pairs:", len(combined_results))
print("Mean:", np.mean(combined_results))
print("Median:", np.median(combined_results))
print("Min:", np.min(combined_results))
print("Max:", np.max(combined_results))

Calculating combined similarity: 100%|█████████████████████████████████████████| 10000/10000 [00:03<00:00, 2645.59it/s]



Finished!
Pairs: 35088
Mean: 78.18058556030817
Median: 82.05923469108053
Min: 14.102564102564102
Max: 100.0


In [20]:
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]

print(f"{'P':<6}{'Name':<12}{'Address':<12}{'Combined':<12}")
print("-" * 42)

for p in percentiles:
    print(
        f"P{p:<4}"
        f"{np.percentile(results, p):<12.2f}"
        f"{np.percentile(address_results, p):<12.2f}"
        f"{np.percentile(combined_results, p):<12.2f}"
    )

P     Name        Address     Combined    
------------------------------------------
P1   8.22        32.96       28.43       
P5   10.53       45.00       45.89       
P10  44.44       51.16       55.01       
P25  71.43       68.49       71.15       
P50  86.21       83.56       82.06       
P75  94.74       92.17       89.47       
P90  100.00      97.83       94.04       
P95  100.00      100.00      96.15       
P99  100.00      100.00      98.94       


In [21]:
weights = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

print(f"{'Name W':<10}{'Address W':<12}{'Mean':<12}{'Median':<12}")
print("-" * 46)

for w in weights:

    a = 1 - w

    scores = [
        w * n + a * ad
        for n, ad in zip(results[:len(address_results)], address_results)
    ]

    print(
        f"{w:<10.1f}"
        f"{a:<12.1f}"
        f"{np.mean(scores):<12.2f}"
        f"{np.median(scores):<12.2f}"
    )

Name W    Address W   Mean        Median      
----------------------------------------------
0.2       0.8         78.63       82.01       
0.3       0.7         78.56       81.86       
0.4       0.6         78.49       81.79       
0.5       0.5         78.42       81.77       
0.6       0.4         78.35       82.14       
0.7       0.3         78.28       83.03       
0.8       0.2         78.21       84.33       


In [22]:
# Make equal-length arrays
n = min(len(results), len(address_results))
nn = min(len(negative_results), len(negative_address_results))

print(f"{'Name W':<10}{'Addr W':<10}{'True Mean':<12}{'Neg Mean':<12}{'Gap':<10}")
print("-" * 54)

for w in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:

    a = 1 - w

    true_scores = (
        w * np.array(results[:n]) +
        a * np.array(address_results[:n])
    )

    neg_scores = (
        w * np.array(negative_results[:nn]) +
        a * np.array(negative_address_results[:nn])
    )

    true_mean = np.mean(true_scores)
    neg_mean = np.mean(neg_scores)

    print(
        f"{w:<10.1f}"
        f"{a:<10.1f}"
        f"{true_mean:<12.2f}"
        f"{neg_mean:<12.2f}"
        f"{true_mean - neg_mean:<10.2f}"
    )

Name W    Addr W    True Mean   Neg Mean    Gap       
------------------------------------------------------
0.2       0.8       78.63       35.05       43.58     
0.3       0.7       78.56       34.73       43.83     
0.4       0.6       78.49       34.42       44.08     
0.5       0.5       78.42       34.10       44.32     
0.6       0.4       78.35       33.78       44.57     
0.7       0.3       78.28       33.46       44.82     
0.8       0.2       78.21       33.14       45.06     


In [23]:
import numpy as np

n = min(len(results), len(address_results))

name_arr = np.array(results[:n])
addr_arr = np.array(address_results[:n])

print("Correlation:", np.corrcoef(name_arr, addr_arr)[0, 1])

Correlation: -0.0038695043811154623


In [24]:
print("Name strong / Address weak:",
      np.mean((name_arr >= 80) & (addr_arr < 50)) * 100, "%")

print("Name weak / Address strong:",
      np.mean((name_arr < 50) & (addr_arr >= 80)) * 100, "%")

print("Both strong:",
      np.mean((name_arr >= 80) & (addr_arr >= 80)) * 100, "%")

print("Both weak:",
      np.mean((name_arr < 50) & (addr_arr < 50)) * 100, "%")

print("\n--- Rescue analysis ---")

print("Average address score when name is weak:",
      addr_arr[name_arr < 50].mean())

print("Average name score when address is weak:",
      name_arr[addr_arr < 50].mean())

Name strong / Address weak: 5.474806201550388 %
Name weak / Address strong: 6.252849977200182 %
Both strong: 36.89295485636115 %
Both weak: 0.9176926584587323 %

--- Rescue analysis ---
Average address score when name is weak: 78.79962945887549
Average name score when address is weak: 77.9401048866519


In [27]:
# Get country directly from the sampled S1 entities
sample_countries = (
    sample["source1_entity_id"]
    .map(source1.set_index("entity_id")["country"])
)

country_results = []

for country, match_ids in zip(
    sample_countries,
    sample["matched_entity_ids"]
):
    country_results.extend(
        [country] * len(str(match_ids).split(","))
    )

n = min(len(country_results), len(results), len(address_results))

country_arr = np.array(country_results[:n])
name_arr2 = np.array(results[:n])
addr_arr2 = np.array(address_results[:n])

for country in ["US", "India"]:

    mask = country_arr == country

    print(f"\n{country}")
    print("Pairs:", mask.sum())
    print("Name mean:", name_arr2[mask].mean())
    print("Address mean:", addr_arr2[mask].mean())
    print("Name median:", np.median(name_arr2[mask]))
    print("Address median:", np.median(addr_arr2[mask]))


US
Pairs: 20933
Name mean: 84.38346085174112
Address mean: 78.67545857668065
Name median: 88.88888888888889
Address median: 83.33333333333334

India
Pairs: 14155
Name mean: 68.71587871164519
Address mean: 78.92748101056587
Name median: 80.95238095238095
Address median: 83.80952380952381
